# ⚡ EV Market Analysis — Final Project Notebook (Europe)

**Made by:** Krish Maisuria, Dev Shah, Miykle Ahmed, Felipe Serna

**Topic:** How do electric vehicle specifications, pricing, and charging infrastructure relate to **innovation**, **affordability**, and **adoption**?

**Europe scope:** We analyze **European-priced EVs** (EUR) and filter the charging station dataset to **Europe only** using a latitude/longitude bounding box (since the stations file does not provide a continent field). Station prices are converted from **USD/kWh → EUR/kWh** for unit consistency.

---
## Research Questions
- **RQ1 (Affordability):** Which brands/models deliver the best value (€/km of range, €/kWh battery)?
- **RQ2 (Innovation):** How do range, battery size, fast charging, and performance relate to price?
- **RQ3 (Adoption proxy):** What do charging station costs, capacity, and installation trends suggest about infrastructure maturity in Europe?
- **RQ4 (Combined):** What is an estimated **energy cost per 100km (EUR)** using EV efficiency and typical EUR/kWh from European stations?


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import plotly.express as px

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Europe bounding box (approximate; continent proxy)
EUROPE_LAT_MIN, EUROPE_LAT_MAX = 34.0, 72.0
EUROPE_LON_MIN, EUROPE_LON_MAX = -25.0, 45.0

# USD -> EUR conversion for station prices (adjust if needed)
USD_TO_EUR = 0.90


## 1) Load Data

**Expected folder structure**
```
COP4283FinalProject/
  notebook.ipynb
  data/
    EV_cars.csv
    detailed_ev_charging_stations.csv
```
If your CSVs are in the same folder as the notebook, it will still work.


In [ ]:
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"

ev_path = DATA_DIR / "EV_cars.csv"
stations_path = DATA_DIR / "detailed_ev_charging_stations.csv"

# Fallback: current directory
if not ev_path.exists():
    ev_path = ROOT / "EV_cars.csv"
if not stations_path.exists():
    stations_path = ROOT / "detailed_ev_charging_stations.csv"

print("EV path:", ev_path)
print("Stations path:", stations_path)

ev_raw = pd.read_csv(ev_path)
stations_raw = pd.read_csv(stations_path)

ev_raw.shape, stations_raw.shape

In [ ]:
display(ev_raw.head())
display(stations_raw.head())

## 2) Cleaning + Feature Engineering

We:
- Standardize column names when needed
- Convert numeric columns
- Remove duplicates and obvious invalid values
- Add derived metrics aligned with the project goals:
  - `kWh_per_100km`
  - `Range_per_kWh_km` (km per kWh, higher is better)
  - `Price_per_kWh_EUR` (€/kWh, lower is better)
  - `Price_per_kmRange_EUR` (€/km of range, lower is better)

**Europe filter:** Charging stations are filtered to Europe using lat/lon bounds, and charging price is converted to **EUR/kWh**.


In [ ]:
def _to_num(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce")

def clean_ev(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    rename_map = {
        "Price.DE.": "Price_EUR",
        "acceleration..0.100.": "Accel_0_100_s",
        "Top_speed": "Top_speed_kmh",
        "Fast_charge": "FastCharge_kmh",
        "Car_name": "Car",
        "Car_name_link": "Link",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    for col in ["Battery", "Efficiency", "FastCharge_kmh", "Price_EUR", "Range", "Top_speed_kmh", "Accel_0_100_s"]:
        if col in df.columns:
            df[col] = _to_num(df[col])

    if "Car" in df.columns:
        df["Brand"] = df["Car"].astype(str).str.strip().str.split().str[0]
    else:
        df["Brand"] = "Unknown"

    df = df.drop_duplicates()

    for col in ["Price_EUR", "Battery", "Range"]:
        if col in df.columns:
            df = df[df[col].isna() | (df[col] > 0)]

    # Efficiency is typically Wh/km => kWh/100km = (Wh/km * 100) / 1000
    df["kWh_per_100km"] = (df["Efficiency"] * 100.0) / 1000.0 if "Efficiency" in df.columns else np.nan
    df["Range_per_kWh_km"] = df["Range"] / df["Battery"]
    df["Price_per_kWh_EUR"] = df["Price_EUR"] / df["Battery"]
    df["Price_per_kmRange_EUR"] = df["Price_EUR"] / df["Range"]

    preferred = [
        "Brand", "Car", "Battery", "Range", "Efficiency", "kWh_per_100km",
        "FastCharge_kmh", "Top_speed_kmh", "Accel_0_100_s",
        "Price_EUR", "Range_per_kWh_km", "Price_per_kWh_EUR", "Price_per_kmRange_EUR",
        "Link"
    ]
    cols = [c for c in preferred if c in df.columns] + [c for c in df.columns if c not in preferred]
    return df[cols]

def clean_stations(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    rename_map = {
        "Station ID": "station_id",
        "Latitude": "lat",
        "Longitude": "lon",
        "Address": "address",
        "Charger Type": "charger_type",
        "Cost (USD/kWh)": "cost_usd_per_kwh",
        "Availability": "availability",
        "Distance to City (km)": "distance_to_city_km",
        "Usage Stats (avg users/day)": "avg_users_per_day",
        "Station Operator": "operator",
        "Charging Capacity (kW)": "capacity_kw",
        "Connector Types": "connector_types",
        "Installation Year": "install_year",
        "Renewable Energy Source": "renewable_source",
        "Reviews (Rating)": "rating",
        "Parking Spots": "parking_spots",
        "Maintenance Frequency": "maintenance_freq",
    }
    df = df.rename(columns={k: v for k, v in rename_map.items() if k in df.columns})

    for col in ["lat", "lon", "cost_usd_per_kwh", "distance_to_city_km", "avg_users_per_day",
                "capacity_kw", "install_year", "rating", "parking_spots"]:
        if col in df.columns:
            df[col] = _to_num(df[col])

    if "address" in df.columns:
        parts = df["address"].astype(str).str.split(",")
        df["city"] = parts.str[-1].str.strip()

    if "charger_type" in df.columns:
        df["is_dc_fast"] = df["charger_type"].astype(str).str.contains("DC", case=False, na=False)

    df = df.drop_duplicates()

    # Validate coordinates
    df = df.dropna(subset=["lat", "lon"])
    df = df[df["lat"].between(-90, 90, inclusive="both") & df["lon"].between(-180, 180, inclusive="both")]

    # ✅ EUROPE-ONLY FILTER (continent proxy)
    df = df[
        df["lat"].between(EUROPE_LAT_MIN, EUROPE_LAT_MAX, inclusive="both")
        & df["lon"].between(EUROPE_LON_MIN, EUROPE_LON_MAX, inclusive="both")
    ]

    # ✅ Convert USD/kWh -> EUR/kWh
    if "cost_usd_per_kwh" in df.columns:
        df["cost_eur_per_kwh"] = df["cost_usd_per_kwh"] * float(USD_TO_EUR)
    else:
        df["cost_eur_per_kwh"] = np.nan

    return df

ev = clean_ev(ev_raw)
stations = clean_stations(stations_raw)

ev.shape, stations.shape

In [ ]:
print("Stations after Europe filter:", len(stations))
display(ev.head())
display(stations.head())

## 3) Summary Statistics (EV Models)

In [ ]:
summary_ev = {
    "n_models": len(ev),
    "median_price_eur": float(ev["Price_EUR"].median()),
    "median_range_km": float(ev["Range"].median()),
    "median_battery_kwh": float(ev["Battery"].median()),
    "median_kwh_per_100km": float(ev["kWh_per_100km"].median()),
}
summary_ev

## 4) Visualizations — Innovation vs Affordability (EUR)

In [ ]:
fig = px.scatter(
    ev.dropna(subset=["Range", "Price_EUR"]),
    x="Range",
    y="Price_EUR",
    color="Brand",
    size="Battery",
    hover_data=["Car", "Battery", "Efficiency", "FastCharge_kmh", "Range_per_kWh_km", "Price_per_kmRange_EUR"],
    labels={"Range": "Range (km)", "Price_EUR": "Price (EUR)", "Battery": "Battery (kWh)"},
    title="EV Price vs Range (color=Brand, size=Battery)"
)
fig.show()

In [ ]:
fig_price = px.histogram(ev, x="Price_EUR", nbins=35, title="Distribution of EV Prices (EUR)")
fig_price.show()

fig_range = px.histogram(ev, x="Range", nbins=35, title="Distribution of EV Ranges (km)")
fig_range.show()

## 5) Charging Station Analysis (Europe only, EUR/kWh)

In [ ]:
stations_summary = {
    "n_stations_europe": len(stations),
    "median_cost_eur_per_kwh": float(stations["cost_eur_per_kwh"].median()),
    "avg_capacity_kw": float(stations["capacity_kw"].mean()),
    "avg_users_per_day": float(stations["avg_users_per_day"].mean()),
}
stations_summary

In [ ]:
fig_cost = px.box(
    stations.dropna(subset=["cost_eur_per_kwh", "charger_type"]),
    x="charger_type",
    y="cost_eur_per_kwh",
    points="outliers",
    title="Charging Cost (EUR/kWh) by Charger Type",
    labels={"charger_type": "Charger type", "cost_eur_per_kwh": "EUR/kWh"}
)
fig_cost.show()

In [ ]:
installs = (stations.dropna(subset=["install_year"])
    .groupby("install_year", as_index=False)
    .size()
    .rename(columns={"size": "stations_installed"})
    .sort_values("install_year")
)

fig_time = px.line(
    installs,
    x="install_year",
    y="stations_installed",
    markers=True,
    title="Charging Stations Installed per Year (Europe Only)",
    labels={"install_year": "Installation year", "stations_installed": "Stations installed"}
)
fig_time.show()

## 6) Combined Insight: Estimated Energy Cost per 100km (EUR)

`EnergyCost_per_100km_EUR = kWh_per_100km * median(EUR/kWh from European stations)`


In [ ]:
median_cost_eur = stations["cost_eur_per_kwh"].median()
median_cost_eur

In [ ]:
combo = ev.copy()
combo["EnergyCost_per_100km_EUR"] = combo["kWh_per_100km"] * float(median_cost_eur)

display(combo[["Brand","Car","kWh_per_100km","EnergyCost_per_100km_EUR","Price_EUR","Range"]].head())

In [ ]:
fig_combined = px.scatter(
    combo.dropna(subset=["Price_EUR","EnergyCost_per_100km_EUR"]),
    x="Price_EUR",
    y="EnergyCost_per_100km_EUR",
    color="Brand",
    hover_data=["Car","Range","Battery","Efficiency","kWh_per_100km"],
    title="Price vs Estimated Energy Cost per 100km (EUR)",
    labels={"Price_EUR":"Price (EUR)", "EnergyCost_per_100km_EUR":"EUR per 100km"}
)
fig_combined.show()

## 7) Limitations & Next Steps

**Limitations**
- Station dataset lacks an explicit continent field, so Europe filtering is done via a lat/lon bounding box.
- Charging prices can vary by membership plans, peak pricing, and regulations across countries.
- EV range is typically rated/estimated; real-world range depends on temperature, speed, and driving style.

**Next steps**
- Add country/region fields (reverse-geocode lat/lon) for more precise European segmentation.
- Use country-level electricity prices and compare across regions.
- Expand to TCO (total cost of ownership) including maintenance and incentives.
